In [36]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVR
from sklearn.ensemble import StackingRegressor

from xgboost import XGBRegressor

In [37]:
df = pd.read_csv(r'data/spotify_tracks.csv')

print('shape:', df.shape)
display(df.head())

shape: (50000, 21)


,track_id,track_name,artist_name,album_name,release_year,genre,popularity,duration_ms,explicit,danceability,...,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,key,mode,time_signature
0,P3fAbnFbmOHnKYaXRvj7uf,One Dance (Acoustic Version),Alex Rodriguez,The Night Album,2024,metal,14,189042,True,0.427723,...,-4.702460,0.050635,0.239506,0.181395,0.133053,0.431384,141.048735,6,0,4
1,M2wleOV911xCZkwPRQeNHp,Forever Song (Remix),Desert Wind,Burning Soul,2019,rock,11,186805,True,0.448634,...,-7.110031,0.000000,0.044463,0.097818,0.435949,0.559135,131.833287,0,1,5
2,4JSnE2NiiUHUAKw9iEU1jj,Last Mountain,The Midnight,The Night Album,2022,k-pop,23,121814,False,0.707923,...,-7.305120,0.144091,0.118380,0.000000,0.262254,0.516873,127.132954,2,1,5
3,2UVvsjaSS8VFgM0Fmxk754,Falling Star (Live),Phantom Keys,Phantom's Greatest Hits,2024,latin,34,216049,False,0.846237,...,-9.527256,0.006668,0.272844,0.000000,0.045332,0.667911,93.041715,0,1,6
4,EeVVhDIq2AnHTmt9OBGhnu,Rising Moon (feat. someone),Desert Wind,Rising Soul,2010,latin,31,156170,False,0.943720,...,-9.017653,0.057632,0.219752,0.098044,0.132083,0.772151,93.404975,7,1,4


In [38]:
X = df.drop(columns=['track_id', 'track_name', 'artist_name', 'album_name', 'popularity'])
y = df['popularity']
print('X shape:', X.shape)

X shape: (50000, 16)


In [39]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)
print('X train columns:', X_train.columns.tolist())

Train shape: (40000, 16)
Test shape: (10000, 16)
X train columns: ['release_year', 'genre', 'duration_ms', 'explicit', 'danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'key', 'mode', 'time_signature']


In [40]:
# preprocessing pipeline
def make_preprocessor(X_data):
    num_cols = X_data.select_dtypes(include=['int64', 'float64']).columns
    cat_cols = X_data.select_dtypes(include=['bool', 'str']).columns

    num_pipline = Pipeline(steps=[('scaler', StandardScaler())])

    cat_pipeline = Pipeline(steps=[('ohe', OneHotEncoder(handle_unknown='ignore'))])

    # column transformer - allows for different pipelines for different columns
    preprocessor = ColumnTransformer(transformers=[('num', num_pipline, num_cols), ('cat', cat_pipeline, cat_cols)])

    return preprocessor

preprocessor = make_preprocessor(X_train)

In [41]:
y_std = y_train.std()
print('y std:', y_std)

y std: 17.916944320439228


In [ ]:
# model pipeline
def make_pipeline(model):
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
    return pipeline

# grouping models and their hyperparameters for grid search
configs = {
    'ElasticNet': {'pipeline': make_pipeline(ElasticNet(random_state=42)), 
                   'params': {'model__alpha': (0.01, .1, 1, 10), 'model__l1_ratio': np.linspace(0, 1, 20)}},
    

    'LinearSVR': {'pipeline': make_pipeline(LinearSVR(random_state=42)), 
            'params': {'model__C': [0.01, .1, 1, 10, 100], 'model__epsilon': [0.01* y_std, 0.05 * y_std, 0.1 * y_std, 0.5 * y_std]}},
    

    'KNN': {'pipeline': make_pipeline(KNeighborsRegressor(random_state=42)), 
            'params': {'model__n_neighbors': np.arange(3, 31), 'model__weights': ['uniform', 'distance']}},
    

    'XGBoost': {'pipeline': make_pipeline(XGBRegressor(objective='reg:squarederror', random_state=42)), 
                'params': {'model__n_estimators': np.arange(100, 201, 25), 'model__learning_rate': [0.01, 0.05, 0.07, 0.1], 
                           'model__max_depth': [3, 5, 7], 'model__subsample': np.linspace(0, 1, 10)}}
    }

In [45]:
# grid search for each model
def tune_gridsearch(model_name, pipeline, params, X_data, y_data):
    print (f'Tuning {model_name}...')

    gs = GridSearchCV(estimator=pipeline, param_grid=params, cv=10, n_jobs=-1, verbose=1, scoring='neg_root_mean_squared_error')

    gs.fit(X_data, y_data)

    print(f'Best params for {model_name}:', gs.best_params_)
    print(f'Best score for {model_name}:', -gs.best_score_)
    
    return gs

gridsearch_results = {}
for model_name, config in configs.items():
    gridsearch_results[model_name] = tune_gridsearch(model_name, config['pipeline'], config['params'], X_train, y_train)

Tuning ElasticNet...
Fitting 10 folds for each of 80 candidates, totalling 800 fits
Best params for ElasticNet: {'model__alpha': 0.1, 'model__l1_ratio': np.float64(1.0)}
Best score for ElasticNet: 17.717579170602033
Tuning LinearSVR...
Fitting 10 folds for each of 20 candidates, totalling 200 fits


c:\Users\aashu\OneDrive\Desktop\Projects\spotify popularity prediction\venv\Lib\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Best params for LinearSVR: {'model__C': 100, 'model__epsilon': np.float64(8.958472160219614)}
Best score for LinearSVR: 17.96068092847519
Tuning KNN...
Fitting 10 folds for each of 56 candidates, totalling 560 fits
Best params for KNN: {'model__n_neighbors': np.int64(30), 'model__weights': 'uniform'}
Best score for KNN: 18.03313368221107
Tuning XGBoost...
Fitting 10 folds for each of 600 candidates, totalling 6000 fits
Best params for XGBoost: {'model__learning_rate': 0.01, 'model__max_depth': 3, 'model__n_estimators': np.int64(200), 'model__subsample': np.float64(0.5555555555555556)}
Best score for XGBoost: 17.723378562927245
